In [2]:
import gradio as gr

def greet(name:str):
    return "Hello " + name + "!"

# Basic example
demo = gr.Interface(fn=greet, inputs="text", outputs="text")
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [3]:
from dotenv import load_dotenv
import os
from openai import OpenAI
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
openai = OpenAI()

In [5]:
#
# This is the function we use to chat with GPT
# Note that, at the time of this exercise, openAPI suggests to use responses
# instead of chat.completions because it allows for more flexibility and tools 
# however
# - the extra tools are mostly paid for and not free
# - the responses api call are openAI only and not compatible with other vendors
#  so overall better to avoid this unless you have specific reasons
system_message = "You are a helpful assistant"
def message_gpt(prompt:str):
     messages = [ { "role": "system", "content": system_message },
                  { "role": "user", "content": prompt } 
                 ]
     response = openai.chat.completions.create(model="gpt-4.1-mini", messages = messages)
     return response.choices[0].message.content


     

In [8]:
#
# thie below shows the training cut off
# no more training after that.. but if you use different models or products you might get different info
print(message_gpt("che giorno e' oggi?"))
#
message_gpt("what date is today?")

Oggi è il 27 aprile 2024. Come posso aiutarti ancora?


'Today is April 27, 2024.'

In [4]:
#
#
#  BUILD A USER INTERFACE
#   - local web server that serve a react UI to interact with a function
#
# This is the function called by the interface
def shout(text:str)->str:
    print(f"Shout has been called with input: {text}")
    return text.upper()
#
# now we create the callback from Gradio
# and just launch gradio with the react frontend
gr.Interface(fn=shout, inputs="textbox",outputs="textbox",flagging_mode="never").launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# if you call the function above with share=True, Gradio will send you code to HuggingFace and 
# it will run on their public servers available for anyone to see (not the best idea for security..) 
# also because the function will still run on your computer (so the local gradio will create an http tunnel
# to the public gradio servers and keep the communicaiton on)
# just don't do it unless you really want to share stuff letting other people use your computer

In [ ]:
#
# you can call launch with: 
#   - inbrowser=True that will automatically open a webbroeser pointing to the function
#   - auth=(<user>,<pwd>) or auth =[(<user1>,<pwd>), ..., ()()] this automatically setup come authentication
#     with user/pwd ot the list of users/pwd able to authenticate

In [14]:
#
# Here we make the application a little nicer
message_input = gr.Textbox(label="Your message:", info="Enter a message to be shouted", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=shout,
    title="Shout", 
    inputs=[message_input],    # this is a list now? it can be both a list and the single value.. I see no difference 
    outputs=[message_output],  # this is a list now?
    examples=["hello", "howdy"], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [18]:
#
# Same as before with message_gpt
#
# This will give you an interactive windows with GPT but:
# - we change system_message to get md formatted response
# - the output textbox is going to be Markdown
#  
system_message = "You are a helpful assistant that responds in markdown without code blocks"
#
message_input = gr.Textbox(label="Your message:", info="Enter a message for GPT-6 Luna", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    #fn=message_gpt,
    fn=stream_gpt,
    title="GPT", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=["hello", "howdy"], 
    flagging_mode="never"
    )
view.launch()


* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


In [ ]:
#
# if you replace the function with a generator, gradle will loop through the response
# until it is completed..
# Here we call the function with stream = True
# The result is comulative
#
# Now go above and replace the function called with this one
def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = openai.chat.completions.create(
        model='gpt-6-luna',
        reasoning_effort='none',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
# You can also add dropdown boxes to select models for instance
#
# Just add:
# - an entry to get the model:
#    model_selector = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
# - then adjust the call to get a model param and call the according model


In [ ]:
#
#
#  AI ASSISTANT CHATBOT
#
# SYSTEM Prompt
# - use the sytem propmt to establish ground rules, like "if you don't know the answer just say so"
# - insert context in the conversation
# - multi shot prompting: provide examples conversations to prime for specific scenarios and train on conversational style 
openai = OpenAI()
MODEL = 'gpt-6-luna'

#
# system message
system_message = "You are a helpful assistant"
#
# callback function that takes 2 params:
#    - message
#    - history
# Note that message and history are automatically populated by Gradio
# and returns a message
def chat(message, history)->str:
    return f"I got the message: '{message}'\ngand the history was:\n{history}"



In [ ]:
#
# now we setup Gradio but this time is a chatInterface
app = gr.ChatInterface(
    fn=chat,
    title="GPT", 
    type="messages"  # <- this specifies that the callback message history should be in a format similar to openAI messages
    ).launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


In [25]:
#
# BETTER CHAT EXAMPLE
#
def chat(message, history):
    # this does a cleanup of history removing the fields we don't care about
    # it makes no difference for openAI (that ignores the extra fields) but it
    # might be a requirement for other models
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    #
    # Gradio will pass only the content of the UI
    # so we need to add the system_message
    # + the history 
    # + the new message as user message
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = openai.chat.completions.create(model=MODEL, 
                                            reasoning_effort="none", 
                                            messages=messages, 
                                            stream=True) # stream is true so this should be a generator
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [26]:
#
# Same sa before
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


In [ ]:
#
# A specific system message will get so much better behaviour and you can select relevant context to add/insert in the system prompt
# so that it can answer specific questions. this is basically the essence of RAG (will be seeen later)
#